In [ ]:
import os
from google.colab import files

# Create the folder
!mkdir -p data

# Upload files into the folder
print("Upload your documents (PDF/TXT) now:")
uploaded = files.upload()
for filename in uploaded.keys():
    os.rename(filename, os.path.join("data", filename))

Upload your documents (PDF/TXT) now:


Saving LenderFeesWorksheetNew.pdf to LenderFeesWorksheetNew.pdf


In [ ]:


import os
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.gemini import Gemini

# 2. Setup your Data
documents = SimpleDirectoryReader("./data").load_data()

# 3. Use MockLLM to avoid OpenAI API Key errors
# This will generate a placeholder response so you can still see the retrieved chunks
os.environ["GOOGLE_API_KEY"] = "URKEY"
Settings.llm = Gemini(model_name="models/gemini-2.5-flash")

# 4. Define the models to test
models = {
    "MiniLM": "sentence-transformers/all-MiniLM-L6-v2",
    "E5": "intfloat/e5-small-v2",
    "BGE": "BAAI/bge-small-en-v1.5"
}

queries = [
    "What is the maximum loan amount a borrower can apply for?",
    "What are the eligibility criteria for the applicant?"
]

# 5. Execution Loop
for name, model_path in models.items():
    print(f"\n{'='*20} Testing Model: {name} {'='*20}")

    # Set the current embedding model
    Settings.embed_model = HuggingFaceEmbedding(model_name=model_path)

    # Re-index for each model to compare retrieval
    index = VectorStoreIndex.from_documents(documents)
    retriever = index.as_retriever(similarity_top_k=2)
    query_engine = index.as_query_engine()

    for q in queries:
        print(f"\n--- Query: {q} ---")

        # This will show a mock response but the real retrieved context
        response = query_engine.query(q)
        print(f"RESPONSE (Mock): {response}")

        # Step 4: Print the Retrieved Chunks (This is the most important part for your task)
        print("\nRETRIEVED CHUNKS:")
        nodes = retriever.retrieve(q)
        for i, node in enumerate(nodes):
            print(f"Chunk {i+1} (Score: {node.score:.4f}):\n{node.get_content()[:300]}...\n")

    print(f"{'='*55}\n")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipython-input-543003747.py:12: DeprecationWarning: Call to deprecated class Gemini. (Should use `llama-index-llms-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/llm/google_genai/This package will no longer be supported after version 0.6.2) -- Deprecated since version 0.6.2.
  Settings.llm = Gemini(model_name="models/gemini-2.5-flash")



==================== Testing Model: MiniLM ====================


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



--- Query: What is the maximum loan amount a borrower can apply for? ---
RESPONSE (Mock): The provided context does not specify the maximum loan amount a borrower can apply for. It only lists a "Total Loan Amount" of $380,000 for a specific example.

RETRIEVED CHUNKS:
Chunk 1 (Score: 0.4662):
Your actual rate, payment, and cost could be higher. Get an official Loan Estimate before choosing a loan.
Fee Details and Summary
Applicants: Application No:
Date Prepared:
Loan Program:
Prepared By:
THIS IS NOT A GOOD FAITH ESTIMATE (GFE). This "Fees W orksheet" is provided for informational purpo...

Chunk 2 (Score: 0.2439):
...


--- Query: What are the eligibility criteria for the applicant? ---
RESPONSE (Mock): The provided context does not contain information regarding the eligibility criteria for the applicant. The document is a "Fees Worksheet" detailing estimated costs and payments for a specific loan application.

RETRIEVED CHUNKS:
Chunk 1 (Score: 0.1522):
Your actual rate, payment, an